In [0]:
# ============================================================
# Allelic dosage interactive dashboard (FULL – one cell)
# CONFIG-DRIVEN (no hardcoded paths or tables)
# ============================================================

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from IPython.display import display, clear_output
import ipywidgets as widgets

# ============================================================
# 0) LOAD CONFIG (single source of truth)
# ============================================================

CONFIG_PATH = "/Volumes/bmqg/default_bronze/fatemeh/config_mixed.yaml"

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)

PATHS = CONFIG["paths"]
DATA  = CONFIG["data"]

GWAS_TABLE  = DATA["gwas_table"]
TAGLO_TABLE = PATHS["TAGLO_TABLE"]
PHENO_PATH  = PATHS["aroma_matrix"]

print("GWAS table :", GWAS_TABLE)
print("TAGLO table:", TAGLO_TABLE)
print("Pheno path :", PHENO_PATH)

# ============================================================
# 1) Plot + data extraction
# ============================================================

def plot_taglo_effect(spark, trait, taglo_ids):
    # ---------- phenotypes ----------
    pheno = pd.read_csv(PHENO_PATH)
    pheno["Variety"] = pheno["Variety"].astype(str).str.strip()
    pheno = pheno[["Variety", trait]].dropna()
    pheno = pheno.rename(columns={trait: "trait_value"})

    # ---------- genotypes (allelic dosage) ----------
    gt = (
        spark.table(TAGLO_TABLE)
        .filter(F.col("taglo_id").isin([int(x) for x in taglo_ids]))
        .select(
            F.col("variety").cast("string").alias("Variety"),
            F.col("value").cast("double").alias("value"),
        )
        .groupBy("Variety")
        .agg(F.round(F.avg("value")).cast("int").alias("dosage"))
        .toPandas()
    )

    # ---------- merge ----------
    df = pheno.merge(gt, on="Variety", how="left")
    df["dosage"] = df["dosage"].fillna(0).astype(int)

    # ---------- plot BEFORE filtering ----------
    plt.figure(figsize=(6,4))
    df.boxplot(column="trait_value", by="dosage")
    plt.title(f"{trait} (raw)")
    plt.suptitle("")
    plt.xlabel("Allelic dosage")
    plt.ylabel(trait)
    plt.tight_layout()
    plt.show()

    return df

# ============================================================
# 2) Manual dosage filter
# ============================================================

def manual_dosage_filter(df, remove_dosages):
    return df[~df["dosage"].isin(set(remove_dosages))].copy()

# ============================================================
# 3) Effect size + fold change
# ============================================================

def compute_effect_and_fc(df, baseline=0, stat="median"):
    g = df.groupby("dosage")["trait_value"]

    if baseline not in g.groups:
        return None, None

    base = getattr(g.get_group(baseline), stat)()

    effects = {
        d: getattr(v, stat)() - base
        for d, v in g
    }

    max_d = max(effects, key=lambda k: abs(effects[k]))
    fc = getattr(g.get_group(max_d), stat)() / base if base != 0 else np.nan

    return effects, fc

# ============================================================
# 4) Build trait → taglo_ids table (from GWAS, config-driven)
# ============================================================

res = (
    spark.table(GWAS_TABLE)
    .filter(F.col("p_wald") < 1e-6)
    .select(
        "trait",
        F.array_distinct(
            F.array("taglo_id1","taglo_id2","taglo_id3","taglo_id4")
        ).alias("taglo_ids")
    )
    .withColumn(
        "taglo_ids",
        F.expr("filter(taglo_ids, x -> x is not null and x > 0)")
    )
    .filter(F.size("taglo_ids") > 0)
    .groupBy("trait")
    .agg(F.first("taglo_ids").alias("taglo_ids"))
    .toPandas()
)

print("Traits available:", len(res))

# ============================================================
# 5) DASHBOARD LOGIC
# ============================================================

def run_trait_dashboard(trait, remove_dosages):
    clear_output(wait=True)

    row = res[res["trait"] == trait].iloc[0]
    taglo_ids = row["taglo_ids"]

    # raw plot
    df = plot_taglo_effect(spark, trait, taglo_ids)

    # summary table
    summary = (
        df.groupby("dosage")["trait_value"]
        .agg(count="count", median="median", mean="mean")
        .reset_index()
        .sort_values("dosage")
    )

    # manual filtering
    df_filt = manual_dosage_filter(df, remove_dosages)
    if df_filt.empty:
        print("❌ No data left after removing dosages")
        print("👉 Please keep at least one dosage (ideally ≥2)")
        display(summary)
        return

    if df_filt["dosage"].nunique() < 2:
        print("❌ Need at least 2 dosage groups to compute fold change / plot")
        display(summary)
        return

    effect, fc = compute_effect_and_fc(df_filt)

    print("Removed dosages:", list(remove_dosages))
    print("Effect (Δ vs baseline):", effect)
    print("Fold change:", fc)

    display(summary)

    # plot AFTER filtering
    plt.figure(figsize=(6,4))
    df_filt.boxplot(column="trait_value", by="dosage")
    plt.title(f"{trait} (after manual dosage removal)")
    plt.suptitle("")
    plt.xlabel("Allelic dosage")
    plt.ylabel(trait)
    plt.tight_layout()
    plt.show()

# ============================================================
# 6) WIDGETS
# ============================================================

trait_dd = widgets.Dropdown(
    options=sorted(res["trait"].tolist()),
    description="Trait:"
)

dosage_ms = widgets.SelectMultiple(
    options=[0, 1, 2, 3, 4],
    description="Remove dosage",
    layout=widgets.Layout(width="260px", height="130px")
)


def update_dosages(change):
    trait = change["new"]
    row = res[res["trait"] == trait].iloc[0]
    df = plot_taglo_effect(spark, trait, row["taglo_ids"])
    dosage_ms.options = sorted(df["dosage"].unique())

trait_dd.observe(update_dosages, names="value")

widgets.interact(
    run_trait_dashboard,
    trait=trait_dd,
    remove_dosages=dosage_ms
);


GWAS table : bmqg.gwas.run_local_20251207
TAGLO table: bmqg.default_silver.taglotype_silver
Pheno path : /Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs/aroma_matrix_GWAS_clean.csv
Traits available: 59


interactive(children=(Dropdown(description='Trait:', options=('(E)-2-Decenal', '(E)-2-Heptenal', '(E, E)-3,5-O…